## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156171?b2cUser=true"><b>Langchain Retrieval PDF</b></a><br/>

Utilizando para fazer pesquisas em documentos para responder perguntas 

<b>PASSOS:</b><br/>
<b>PASSO 1 - CARGA NO CARREGADOR</b><br/>
<b>PASSO 2 - CRIAÇÃO DO ÍNDICE DE BUSCA</b><br/>
<ul><li><b>2.1 - QUEBRA DO TEXTO</b></li></ul>
<ul><li><b>2.2 - INDEXANDO AS QUEBRAS DO TEXTO</b></li></ul>
<ul><li><b>2.3 - ARMAZENANDO OS ÍNDICES EM UM BANCO VETORIAL NA MEMÓRIA</b></li></ul>
<b>PASSO 3 - EXECUTANDO A PESQUISA</b>

In [1]:
#%pip install -r requirements.txt

#### <b>EXEMPLO 1</b><br/>
O objetivo será realizar a pesquisa em um arquivo txt, sobre os benefícios do cartão Gold contra roubo.

In [2]:
from langchain_openai import ChatOpenAI
from os import getenv
from dotenv import load_dotenv # CARREGA A VARIÁVEL DE AMBIENTE OPENAI_KEY LIDA DO ARQUIVO .env
from langchain_core.globals import set_debug

set_debug(True)

load_dotenv() # CARREGANDO O ARQUIVO COM A OPENAI_KEY

llm = ChatOpenAI( # INSTANCIANDO A LLM
                    model="gpt-5-mini",                    
                    # 1 - OBTENDO A API KEY POR MEIO DA VARIÁVEL DE AMBIENTE OPENAI_KEY. QUE VAI FICAR ARMAZENADA NO ARQUIVO .env.
                    # 2 - AINDA É NECESSÁRIO CARREGAR ESSE ARQUIVO. VER NA PRIMEIRA CÉLULA DO NOTEBOOK
                    api_key=getenv("OPENAI_KEY")                    
                )

#### <b>PASSO 1 - CARGA NO CARREGADOR</b>

Carregando vários documentos

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# CRIAÇÃO
carregadores = [
                    PyPDFLoader("../documentos/GTB_gold_Nov23.pdf"),
                    PyPDFLoader("../documentos/GTB_platinum_Nov23.pdf"),
                    PyPDFLoader("../documentos/GTB_standard_Nov23.pdf"),
                ]

# CARGA NO CARREGADOR
documentos = []
for carregador in carregadores:
    print('Carregando documento: ',carregador.source)
    documentos.append(carregador.load()) # UM CARREGADOR DEVOLVE UM ARRAY CONTENDO O DOCUMENTO. AQUI TEMOS UM ARRAY DE ARRAYS.
    print('Documento carregado\n',documentos[-1],'\n')
    

Carregando documento:  ../documentos/GTB_gold_Nov23.pdf
Documento carregado
 [Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T10:05:20-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:24:39Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '8050f60f-7cb7-4b1b-99e0-7f18410feb72', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T10:05:20-03:00', 'source': '../documentos/GTB_gold_Nov23.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='1 

#### <b>PASSO 2 - CRIAÇÃO DO ÍNDICE DE BUSCA</b>

<li>Para isso, será necessário, primeiramente, realizar a quebra (splitter) em trechos, para que a IA possa indexá-los.

<b>2.1 - QUEBRA DO TEXTO</b>

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# DEFINIÇÃO DO QUEBRADOR
quebrador = CharacterTextSplitter(chunk_size=1000,chunk_overlap=200) # QUEBRANDO EM CARACTERES. DE 1000 EM 1000 CARACTERES.
                                                                     # CHUNK_OVERLAP: SOBREPOSIÇÃO DO TEXTO
# TRANSFORMANDO O ARRAY DE ARRAYS EM APENAS UM ARRAY
documentos = [
                item 
                for sublist in documentos 
                    for item in sublist
            ]  

# QUEBRA DO TEXTO EM VÁRIOS TEXTOS
textos = quebrador.split_documents(documentos) # AQUI, DOCUMENTOS NÃO PODE SER UM ARRAY DE ARRAYS. TEM QUE SER APENAS UM ARRAY DE DOCUMENTOS.

print('\nTextos\n',textos)


Textos
 [Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T10:05:20-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:24:39Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '8050f60f-7cb7-4b1b-99e0-7f18410feb72', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T10:05:20-03:00', 'source': '../documentos/GTB_gold_Nov23.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Ediç

<b>2.2 - INDEXANDO AS QUEBRAS DE TEXTO</b>

<ol>
    <li>Sequências de palavras que possuem sentido semelhante, terão números semelhantes no espaço</li>
    <li>Esse números são chamados de índices, e esses <b>índices</b> recebem o nome de <b>embeddings</b></li>
</ol>

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

# CRIANDO OS EMBEDDINGS
#embeddings = OpenAIEmbeddings(api_key=getenv("OPENAI_KEY"))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print('Embeddings\n',embeddings)

Embeddings
 model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={'device': 'cpu'} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


<b>2.3 - ARMAZENANDO OS ÍNDICES EM UM BANCO VETORIAL NA MEMÓRIA</b>

In [23]:
from langchain.vectorstores import FAISS

# LUGAR PARA ARMAZENAR OS EMBEDDINGS -> BANCO VETORIAL. ARMAZENA O NÚMERO E A FRASE.
# SERÁ USADO O BANCO VETORIAL FAISS DO Facebook
db = FAISS.from_documents(textos,embeddings)   # CRIADO O BD A PARTIR DOS DOCUMENTOS

print(db)


#### <b>PASSO 3 - EXECUTANDO A PESQUISA</b>

In [35]:
from langchain.chains import RetrievalQA

# create the RetrievalQA chain using the existing llm and the retriever (Quem busca no banco de dados)
# qa_chain -> Nossa ferramenta de Perguntas e Respostas (Questions and Answers Chain)
qa_chain = RetrievalQA.from_chain_type(
                                        llm=llm, 
                                        retriever=db.as_retriever(),
                                        return_source_documents=True
                                      )

# exemplo de uso
pergunta = "Como devo proceder caso tenha um item pessoal roubado ?. Não faça qualquer tipo de comentário ou pergunta, apenas responda a pergunta."

resposta = qa_chain.invoke({"query": pergunta})
print('\nPergunta: ',pergunta,'\nResposta\n', resposta['result'],
      '\nDocumentos de origem:\n',resposta['source_documents'])


[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Como devo proceder caso tenha um item pessoal roubado ?. Não faça qualquer tipo de comentário ou pergunta, apenas responda a pergunta."
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Como devo proceder caso tenha um item pessoal roubado ?. Não faça qualquer tipo de comentário ou pergunta, apenas responda a pergunta.",
  "context": "19 \nVersão: novembro 2023 \n2021 \n \n \nsinistro. É sua responsabilidade informar esses dados para processamento do sinistro. \n \n* É possível que seja solicitado ao Portador de Cartão que envie o item ou itens danificados, às suas custas, \npara melhor avaliação da reivindicação. \n \n \n \n \nPROTEÇÃO DE PREÇOS * \n \nInformações Exigidas (comprovante de perdas): \n \n1) Um recibo original mostr